In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path

MODELS_DIR = Path("/kaggle/input/datasets/sqizeeeeeeeee/dataset/models/models")
MODELS_NPZ_DIR = Path("/kaggle/input/datasets/sqizeeeeeeeee/dataset/models_npz/models_npz")

print(list(MODELS_DIR.glob("*")))
print(list(MODELS_NPZ_DIR.glob("*")))

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pkl", ".txt", ".npz")):
            print(os.path.join(root, f))

In [ ]:
from pathlib import Path
import pandas as pd

TEST_PATH = Path("/kaggle/input/competitions/llm-classification-finetuning/test.csv")

df = pd.read_csv(TEST_PATH)
print(df.shape)
df.head(3)

In [ ]:
from ast import literal_eval

def safe_parse(x):
    try:
        parsed = literal_eval(x)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass
    return [x]

def sanitize_text(text):
    return text.encode("utf-8", errors="surrogatepass").decode("utf-8", errors="replace")

def parse_text_columns(df):
    for col in ["prompt", "response_a", "response_b"]:
        parsed = df[col].apply(safe_parse)
        df[f"{col}_text"] = parsed.apply(lambda turns: sanitize_text(" ".join(turns).strip()))
    return df

df = parse_text_columns(df)
df[["prompt_text", "response_a_text", "response_b_text"]].head(2)

In [ ]:
REL_LEN_CLIP = 50

def add_length_features(df):
    df["prompt_len_chars"] = df["prompt_text"].str.len()
    df["response_a_len_chars"] = df["response_a_text"].str.len()
    df["response_b_len_chars"] = df["response_b_text"].str.len()
    df["len_diff"] = df["response_a_len_chars"] - df["response_b_len_chars"]

    df["a_rel_len"] = df["response_a_len_chars"] / (df["prompt_len_chars"] + 1)
    df["b_rel_len"] = df["response_b_len_chars"] / (df["prompt_len_chars"] + 1)
    rel_len_diff = df["a_rel_len"] - df["b_rel_len"]
    df["rel_len_diff"] = rel_len_diff.clip(-REL_LEN_CLIP, REL_LEN_CLIP)
    return df

df = add_length_features(df)
df[["len_diff", "rel_len_diff"]].head(3)

In [ ]:
import re

def count_markdown_features(text):
    return {
        "n_bullets": len(re.findall(r"(?:^|\n)[\-\*]\s", text)),
        "n_numbered": len(re.findall(r"(?:^|\n)\d+\.\s", text)),
        "n_headers": len(re.findall(r"(?:^|\n)#{1,6}\s", text)),
        "n_bold": len(re.findall(r"\*\*[^*]+\*\*", text)),
        "n_code_blocks": text.count("```"),
    }

def add_formatting_features(df):
    feat_a = df["response_a_text"].apply(count_markdown_features).apply(pd.Series).add_prefix("a_")
    feat_b = df["response_b_text"].apply(count_markdown_features).apply(pd.Series).add_prefix("b_")
    df = pd.concat([df, feat_a, feat_b], axis=1)

    for feat in ["n_bullets", "n_numbered", "n_headers", "n_bold", "n_code_blocks"]:
        df[f"{feat}_diff"] = df[f"a_{feat}"] - df[f"b_{feat}"]
    return df

df = add_formatting_features(df)
df[[c for c in df.columns if c.endswith("_diff") and "len" not in c]].head(3)

In [ ]:
REFUSAL_PATTERN = re.compile(
    r"\b(?:i cannot|i can't|i'm sorry|i am sorry|as an ai|i'm not able to|"
    r"i don't have the ability|i'm unable to|cannot provide|can't provide)\b",
    flags=re.IGNORECASE,
)

def add_refusal_features(df):
    df["a_refusal"] = df["response_a_text"].str.contains(REFUSAL_PATTERN, regex=True).astype(int)
    df["b_refusal"] = df["response_b_text"].str.contains(REFUSAL_PATTERN, regex=True).astype(int)
    df["refusal_diff"] = df["a_refusal"] - df["b_refusal"]
    return df

df = add_refusal_features(df)
df[["a_refusal", "b_refusal", "refusal_diff"]]

In [ ]:
import lightgbm as lgb

FEATURE_COLS = [
    "prompt_len_chars", "response_a_len_chars", "response_b_len_chars", "len_diff",
    "a_rel_len", "b_rel_len", "rel_len_diff",
    "a_n_bullets", "a_n_numbered", "a_n_headers", "a_n_bold", "a_n_code_blocks",
    "b_n_bullets", "b_n_numbered", "b_n_headers", "b_n_bold", "b_n_code_blocks",
    "n_bullets_diff", "n_numbered_diff", "n_headers_diff", "n_bold_diff", "n_code_blocks_diff",
    "a_refusal", "b_refusal", "refusal_diff",
]

X = df[FEATURE_COLS].values
print(X.shape)

def logreg_predict_proba_from_npz(npz_path, X):
    data = np.load(npz_path)
    coef = data["coef"]
    intercept = data["intercept"]
    classes = data["classes"]
    scaler_mean = data["scaler_mean"]
    scaler_scale = data["scaler_scale"]

    X_scaled = (X - scaler_mean) / scaler_scale
    logits = X_scaled @ coef.T + intercept

    logits -= logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    proba = exp_logits / exp_logits.sum(axis=1, keepdims=True)

    return proba, classes

booster0 = lgb.Booster(model_file=str(MODELS_DIR / "lgbm_fold0.txt"))

proba0, classes0 = logreg_predict_proba_from_npz(MODELS_NPZ_DIR / "logreg_fold0.npz", X)
print(classes0)
print(proba0)
print(booster0.predict(X, num_iteration=booster0.best_iteration))

In [ ]:
N_FOLDS = 5
LGBM_WEIGHT = 0.95
CLASS_NAMES = ["a", "b", "tie"]

logreg_preds = []
lgbm_preds = []

for fold in range(N_FOLDS):
    proba, _ = logreg_predict_proba_from_npz(MODELS_NPZ_DIR / f"logreg_fold{fold}.npz", X)
    logreg_preds.append(proba)

    booster = lgb.Booster(model_file=str(MODELS_DIR / f"lgbm_fold{fold}.txt"))
    lgbm_pred = booster.predict(X, num_iteration=booster.best_iteration)
    lgbm_preds.append(lgbm_pred)

    print(f"[fold {fold}] predicted")

logreg_avg = np.mean(logreg_preds, axis=0)
lgbm_avg = np.mean(lgbm_preds, axis=0)

final_pred = LGBM_WEIGHT * lgbm_avg + (1 - LGBM_WEIGHT) * logreg_avg

submission = pd.DataFrame({
    "id": df["id"],
    "winner_model_a": final_pred[:, CLASS_NAMES.index("a")],
    "winner_model_b": final_pred[:, CLASS_NAMES.index("b")],
    "winner_tie": final_pred[:, CLASS_NAMES.index("tie")],
})

submission.to_csv("submission.csv", index=False)
print(submission)